<a href="https://colab.research.google.com/github/firdoushkhilji/firdoushkhilji-7006SCN_FK_16943920/blob/task4/Task4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# Task 4 — Distributed Computing

!pip install pyspark -q

from pyspark.sql import SparkSession
import time

# RESOURCE CONFIGURATION
# These settings control how much memory Spark allocates
# and how data is distributed across partitions

spark = SparkSession.builder \
    .appName("NHS_Task4_DistributedComputing") \
    .config("spark.driver.memory", "8g") \
    .config("spark.executor.memory", "4g") \
    .config("spark.sql.shuffle.partitions", "8") \
    .config("spark.default.parallelism", "8") \
    .getOrCreate()

print("SparkSession started with custom resource configuration!")

print("\nRESOURCE CONFIGURATION")
print("-" * 50)
print(f"Driver Memory: {spark.conf.get('spark.driver.memory')}")
print(f"Executor Memory: {spark.conf.get('spark.executor.memory')}")
print(f"Shuffle Partitions: {spark.conf.get('spark.sql.shuffle.partitions')}")
print(f"Default Parallelism: {spark.sparkContext.defaultParallelism}")
print("-" * 50)

SparkSession started with custom resource configuration!

RESOURCE CONFIGURATION
--------------------------------------------------
Driver Memory: 8g
Executor Memory: 4g
Shuffle Partitions: 8
Default Parallelism: 8
--------------------------------------------------


In [5]:
from google.colab import drive
drive.mount('/content/drive')

#load Dataset
train_df = spark.read.parquet(
    '/content/drive/MyDrive/TASK_DATASET/train_data.parquet'
)
test_df = spark.read.parquet(
    '/content/drive/MyDrive/TASK_DATASET/test_data.parquet'
)

print("\nDataset Loaded Successfully!")
print(f"Training rows: {train_df.count():,}")
print(f"Testing rows: {test_df.count():,}")

# ACCESS SPARK UI
# Spark UI runs on localhost but Colab needs ngrok/proxy to view it
# Install pyngrok to create a public tunnel to view Spark UI
!pip install pyngrok -q

from pyngrok import ngrok

# Paste your authtoken here
ngrok.set_auth_token("3FSdkYhtTHZqHjhHMQLzt3lFSiF_3KqpRuj7FzskjeD78LWrn")

print("Ngrok authenticated!")

# Spark UI runs on port 4040 by default
public_url = ngrok.connect(4040)
print(f"Spark UI available at: {public_url}")
print("Click this link to view the Spark UI Jobs and Storage tabs")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

Dataset Loaded Successfully!
Training rows: 1,469,696
Testing rows: 367,299


Ngrok authenticated!
Spark UI available at: NgrokTunnel: "https://egomaniac-errant-stumbling.ngrok-free.dev" -> "http://localhost:4040"
Click this link to view the Spark UI Jobs and Storage tabs


In [6]:
# REPARTITIONING STRATEGY
# Check current partition count
print(f"Current partitions: {train_df.rdd.getNumPartitions()}")

# Repartition to match our 8-core configuration
# Too few partitions = underutilised parallelism (cores sit idle)
# Too many partitions = excessive task scheduling overhead
train_df_repartitioned = train_df.repartition(8)
test_df_repartitioned = test_df.repartition(8)

print(f"Partitions after repartitioning: {train_df_repartitioned.rdd.getNumPartitions()}")

# Verify partition balance
from pyspark.sql.functions import spark_partition_id

print("\nPartition size distribution:")
train_df_repartitioned.groupBy(spark_partition_id().alias("partition_id")) \
    .count() \
    .orderBy("partition_id") \
    .show()

Current partitions: 8
Partitions after repartitioning: 8

Partition size distribution:
+------------+------+
|partition_id| count|
+------------+------+
|           0|183712|
|           1|183710|
|           2|183712|
|           3|183711|
|           4|183712|
|           5|183713|
|           6|183713|
|           7|183713|
+------------+------+



##REPARTITIONING STRATEGY EXPLANATION:

###WHAT IS REPARTITIONING?
Spark splits data into "partitions" — chunks that can be processed
in parallel across available CPU cores. Repartitioning controls
how many of these chunks exist and how data is distributed among
them.

###WHY IT MATTERS:
- Too FEW partitions: If you have eight separate cores, but only two partitions are present,
  you essentially have six idle, and two cores are doing all the work.
  This is a waste of the available parallelism.
- Too MANY partitions: If you had eight cores, but instead of creating only eight partitions
  you create 1,000 partitions which are very small, Spark doesn't actually process the data,
  instead it is spending time just scheduling and coordinating the running of the tasks.
  The overhead caused by this excessive scheduling far exceeds the benefits gained from processing the data.

###OUR DECISION:
To ensure our SparkSession was configured correctly for using eight partitions versus the eight parallel
processing units, and to ensure max throughput without excessive scheduling, we repartitioned to eight partitions,
thus creating eight partitions so that each of the eight logical cores had only one partition available
to be worked on concurrently.

###VERIFICATION:
Using spark_partition_id() we formatted our data to see how many rows were in each of the partitions created by our
job to determine if our data was evenly distributed across partitions (good) or whether there were very large and
very small partitions (bad) which could result in straggler tasks being created which would slow down the entire job.

###WHY THIS MATTERS FOR OUR PROJECT:
With a total of 1,469,696 training rows processed across the eight partitions, the average number of rows processed
by each partition is approximately 183,712 (a manageable sized amount) which provides a balance between the benefits
from parallel processing and the overhead per task, especially when you consider the number of times that CrossValidator
was used during task 3 to read the partitioned data multiple times across the different folds using different
parameter combinations.


In [7]:
# CACHING STRATEGY
# CrossValidator in Task 3 read train_sample multiple times:
# - 4 ParamGrid combinations × 2 folds = 8 reads minimum per model
# - Across 4 models = up to 32 total reads of the same data
# Without caching, Spark would re-read from Parquet disk EVERY time
# Caching keeps data in memory after the first read, avoiding
# repeated disk I/O for subsequent reads

from pyspark import StorageLevel

# MEMORY_AND_DISK: tries memory first, spills to disk if memory full
# Safer than MEMORY_ONLY for our Colab environment with limited RAM
train_df_repartitioned.persist(StorageLevel.MEMORY_AND_DISK)

# Trigger the cache by running an action
row_count = train_df_repartitioned.count()
print(f"Cached {row_count:,} rows using MEMORY_AND_DISK storage level")

Cached 1,469,696 rows using MEMORY_AND_DISK storage level


###CACHING JUSTIFICATION:

train_df_repartitioned was cached using MEMORY_AND_DISK storage
level after repartitioning. This DataFrame is read repeatedly
during CrossValidator hyperparameter tuning — with 4 ParamGrid
combinations and 2-fold cross-validation, Spark accesses the
training data up to 8 times per model. Across 4 models in Task 3,
this represents up to 32 total reads.

Without caching, each read would require re-parsing the Parquet
file and reapplying any transformations from disk — a significant
overhead at scale. MEMORY_AND_DISK was selected over MEMORY_ONLY
because it gracefully spills to disk if the cached data exceeds
available RAM, preventing OutOfMemoryError crashes — a critical
consideration given Colab's 12GB RAM limit, which caused multiple
crashes during Task 3 model training before sample-size reduction.